# 🧹 Notebook 02: Text Preprocessing & Label Generation

**Project:** Shopee App Review Intelligence Dashboard  
**Input Dataset:** `data/raw/scrapped_Shopee 12.12.csv` (85.500 baris)  
**Output Dataset:** `data/processed/shopee_reviews_clean.csv`  

---
## 🎯 Tahapan Preprocessing Teks
1. **Drop Null & Data Cleaning**: Menghapus data null pada kolom review `content`.
2. **Case Folding**: Menyeragamkan seluruh teks menjadi huruf kecil (*lowercase*).
3. **Pemisahan Tanda Baca & Regex Cleaning**: Mengganti tanda baca dengan spasi agar kata-kata tidak menempel (contoh: *"sayang,ongkir"* -> *"sayang ongkir"*).
4. **Normalisasi Kata Ulang (Angka 2)**: Menangani kata ulang seperti *"kadang2"* -> *"kadang kadang"*.
5. **Elongation Normalization**: Memperbaiki huruf berulang berlebihan (misal *"bagusssss"* -> *"bagus"*).
6. **Slang Dictionary Normalization (Bahasa Indonesia)**: Memetakan kata gaul/singkatan (*bpk -> bapak*, *kk -> kakak*, *min -> admin*, *telp -> telepon*, *gk -> tidak*, *bgt -> banget*, *sdh -> sudah*, *klo -> kalau*, *ongkir -> ongkos kirim*, *d -> di*, *tko -> toko*, dsb.).
7. **Selective Stopwords Removal (Preservasi Kata Konteks & Negasi)**: Menghapus hanya kata hubung gramatikal murni (*yang, di, ke, dari, dan, atau, ini, itu*) sambil mempertahankan kata kontekstual (*bisa, kita, ada, lagi, buat, biar, tolong, bapak, ibu, admin*) dan negasi (*tidak, bukan, kurang*).
8. **Dokumentasi Keputusan Stemming Sastrawi**: Evaluasi dampak stemming terhadap performa dan preservasi makna teks.
9. **Sentiment Proxy Labeling**: Pemetaan rating `score` 1-2 (`negatif`), 3 (`netral`), 4-5 (`positif`).


In [1]:
import os
import re
import pandas as pd
import numpy as np
import time

raw_path = os.path.join("..", "data", "raw", "scrapped_Shopee 12.12.csv")
df = pd.read_csv(raw_path)

print(f"Raw dataset shape: {df.shape}")
df = df.dropna(subset=['content']).copy()
print(f"After dropping null content: {df.shape[0]:,} rows")


Raw dataset shape: (85500, 11)
After dropping null content: 85,499 rows


In [2]:
# 1. Expanded Slang Normalization Dictionary
slang_dict = {
    'bpk': 'bapak', 'pak': 'bapak',
    'kk': 'kakak', 'kak': 'kakak',
    'min': 'admin', 'adm': 'admin',
    'telp': 'telepon', 'tlp': 'telepon', 'telfon': 'telepon', 'telpon': 'telepon',
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'ngga': 'tidak', 'nggak': 'tidak', 'g': 'tidak',
    'bgt': 'banget', 'bangt': 'banget',
    'klo': 'kalau', 'kalo': 'kalau', 'kl': 'kalau',
    'sdh': 'sudah', 'udh': 'sudah', 'dh': 'sudah',
    'tp': 'tapi', 'tpi': 'tapi',
    'jd': 'jadi', 'jdi': 'jadi',
    'lg': 'lagi', 'lgi': 'lagi',
    'blm': 'belum', 'belom': 'belum',
    'sy': 'saya', 'sya': 'saya',
    'dpt': 'dapat', 'dpet': 'dapat',
    'dri': 'dari', 'org': 'orang',
    'pake': 'pakai', 'pakek': 'pakai',
    'bkn': 'bukan', 'sm': 'sama', 'yg': 'yang', 'utk': 'untuk', 'dlm': 'dalam',
    'aja': 'saja', 'aj': 'saja', 'skrg': 'sekarang', 'brg': 'barang',
    'ori': 'original', 'ongkir': 'ongkos kirim', 'ongkirnya': 'ongkos kirimnya',
    'cs': 'customer service', 'eror': 'error', 'apk': 'aplikasi', 'lemot': 'lambat', 'parah': 'parah',
    'seller': 'penjual', 'fast': 'cepat', 'respon': 'respon',
    'nyesel': 'menyesal', 'moga': 'semoga', 'makasih': 'terima kasih', 'mksh': 'terima kasih',
    'emang': 'memang', 'emng': 'memang', 'cuman': 'cuma', 'cmn': 'cuma',
    'tko': 'toko', 'd': 'di', 'laa': 'lah', 'la': 'lah', 'si': 'sih',
    'tambahin': 'tambahkan', 'nyaa': 'nya'
}

# 2. Selective Stopwords (Mempertahankan kata konteks seperti 'bisa', 'kita', 'bapak', 'ibu', 'admin', 'lagi', 'buat', 'tolong')
pure_stopwords = {
    'yang', 'di', 'ke', 'dari', 'dan', 'atau', 'ini', 'itu', 'pada', 'oleh', 'adalah', 'yaitu', 'adapun',
    'dengan', 'untuk', 'tersebut', 'bahwa', 'serta'
}

print(f"Custom selective stopwords count: {len(pure_stopwords)}")


Custom selective stopwords count: 18


In [3]:
url_pattern = re.compile(r'https?://\S+|www\.\S+')
html_pattern = re.compile(r'<.*?>')
elongation_pattern = re.compile(r'(.)\1{2,}')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = url_pattern.sub('', text)
    text = html_pattern.sub('', text)
    text = re.sub(r'(\w+)2', r'\1 \1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = elongation_pattern.sub(r'\1', text)
    
    tokens = text.split()
    cleaned_tokens = []
    for token in tokens:
        token = slang_dict.get(token, token)
        if token not in pure_stopwords and len(token) > 0:
            cleaned_tokens.append(token)
    return ' '.join(cleaned_tokens)

print("Menjalankan pipeline preprocessing teks...")
start_t = time.time()
df['clean_content'] = df['content'].apply(preprocess_text).fillna('')
print(f"Selesai dalam {time.time() - start_t:.2f} detik!")


Menjalankan pipeline preprocessing teks...


Selesai dalam 3.25 detik!


In [4]:
sample_df = df[['content', 'clean_content', 'score']].head(10)
print("=== PERBANDINGAN SAMPEL TEKS ASLI VS CLEAN CONTENT ===")
for idx, row in sample_df.iterrows():
    print(f"Rating : {row['score']} ⭐")
    print(f"Asli   : {row['content']}")
    print(f"Bersih : {row['clean_content']}")
    print("-" * 60)


=== PERBANDINGAN SAMPEL TEKS ASLI VS CLEAN CONTENT ===
Rating : 1 ⭐
Asli   : penipuan berkedok Flash sale,ga ada mati nya,IKLAN AJA DI TV DAH KAYA YG BENERAN,DAN TERNYATA,ASU.tapi SESUAILAH sama Tagline nya,yg di iklan kan,sm Artisnya,pasti murah,karna udh ga mampu endorse ARTIS MAHAL,akhir nya,ya sesuai Tagline nya.ahayyyyyyy,...LAGI JATUH ne ye.
Bersih : penipuan berkedok flash sale tidak ada mati nya iklan saja tv dah kaya beneran ternyata asu tapi sesuailah sama tagline nya iklan kan sama artisnya pasti murah karna sudah tidak mampu endorse artis mahal akhir nya ya sesuai tagline nya ahay lagi jatuh ne ye
------------------------------------------------------------
Rating : 4 ⭐
Asli   : lama kali kurir nya , kalo bisa min... , tambahin laa identitas kurir nyaa.. biar kita bisa telp/chat kurir nya
Bersih : lama kali kurir nya kalau bisa admin tambahkan lah identitas kurir nya biar kita bisa telepon chat kurir nya
------------------------------------------------------------
Rating : 

### 💡 Evaluasi & Keputusan Preprocessing & Stemming

> **1. Preservasi Kata Kontekstual & Negasi:**  
> Penghapusan stopword dilakukan secara selektif (*selective stopwords removal*). Kata-kata seperti *"bisa"*, *"kita"*, *"bapak"*, *"ibu"*, *"admin"*, *"lagi"*, *"buat"*, *"tolong"*, *"biar"* dipertahankan karena membawa konteks pembicaraan dan keluhan pengguna secara utuh.  

> **2. Penanganan Tanda Baca & Kata Ulang:**  
> Tanda baca diganti dengan spasi agar kata yang ditulis tanpa spasi (misal *"sayang,ongkir"* -> *"sayang ongkir"*) tidak menempel menjadi satu token tidak valid. Angka pengulangan kata (seperti *"kadang2"*) diubah menjadi *"kadang kadang"*.  

> **3. Keputusan Stemming Sastrawi:**  
> Stemming dilewati (*skipped*) untuk menjaga keutuhan derajat kata sentimen (seperti *"penipuan"* vs *"tipu"*) dan mempercepat komputasi (selesai dalam ~1,6 detik).


In [5]:
df['sentiment'] = df['score'].apply(lambda s: 'negatif' if s <= 2 else ('netral' if s == 3 else 'positif'))

print("=== DISTRIBUSI KELAS SENTIMEN PROXY ===")
dist = df['sentiment'].value_counts()
pct = (dist / len(df) * 100).round(2)
for label, count in dist.items():
    print(f"{label.capitalize():<8}: {count:>6,} review ({pct[label]:>5.2f}%)")

print(f"\nNull check pada clean_content: {df['clean_content'].isnull().sum()}")


=== DISTRIBUSI KELAS SENTIMEN PROXY ===
Positif : 40,122 review (46.93%)
Negatif : 36,837 review (43.08%)
Netral  :  8,540 review ( 9.99%)

Null check pada clean_content: 0


In [6]:
out_path = os.path.join("..", "data", "processed", "shopee_reviews_clean.csv")
df.to_csv(out_path, index=False, encoding='utf-8')
print(f"Dataset bersih berhasil disimpan ke: {out_path}")
print(f"Ukuran File: {os.path.getsize(out_path) / (1024*1024):.2f} MB")


Dataset bersih berhasil disimpan ke: ..\data\processed\shopee_reviews_clean.csv
Ukuran File: 72.90 MB
